# Берем готовый датасет lstm и с помощью bigquery собираем датасет для gnn

In [1]:
# Установить клиентскую библиотеку
%pip install google-cloud-bigquery pandas


  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached requests-2.32.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached charset_normalizer-3.4.2-cp312-cp312-win_amd64.whl.metadata (36 kB)
  Using cached idna-3.10-py3-none-any.whl.metadata (10 kB)
  Using cached urllib3-2.4.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached certifi-2025.4.26-py3-none-any.whl.metadata (2.5 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
  Using cached numpy-2.2.5-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached cachetools-5.5.2-py3-none-any.whl (10 kB)
   ---------------------------------------- 0.0/4.3 MB ? eta -:--:--
 

In [6]:
# Шаг 1: Авторизация (если не делал)
!gcloud auth application-default login --quiet

# Шаг 2: Установка проекта для квоты
!gcloud auth application-default set-quota-project celtic-tendril-459507-q8

"gcloud" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.
"gcloud" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [30]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:\\Users\\fesevu\\AppData\\Roaming\\gcloud\\application_default_credentials.json"
# замените на свой ID
os.environ["GOOGLE_CLOUD_PROJECT"] = "celtic-tendril-459507-q8"

# Загружает LSTM датасет


In [7]:
import pandas as pd
from google.cloud import bigquery

# 1. Считываем LSTM-датасет
lstm_df = pd.read_csv('./data/transaction_dataset.csv', usecols=['Address','FLAG'])
lstm_df.columns = ['id','label']
seed_addrs = set(lstm_df['id'])

# Старая версия

In [ ]:
import pandas as pd

# Читаем LSTM-адреса
lstm = pd.read_csv('./data/transaction_dataset.csv', usecols=['Address'])
addrs = lstm['Address'].unique().tolist()

# 2. Формируем строки для UNNEST
addr_lines = [f"    '{addr}'" for addr in addrs]
addr_block = ",\n".join(addr_lines)
# 3. Шаблон SQL-запроса с JOIN вместо IN-подзапросов
sql = f"""#standardSQL
WITH
  seed AS (
    SELECT addr AS node
    FROM UNNEST([
{addr_block}
    ]) AS addr
  ),
  level1_out AS (
    SELECT t.to_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN seed ON t.from_address = seed.node
  ),
  level1_in AS (
    SELECT t.from_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN seed ON t.to_address = seed.node
  ),
  level1 AS (
    SELECT node FROM level1_out
    UNION DISTINCT
    SELECT node FROM level1_in
  ),
  level2_out AS (
    SELECT t.to_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN level1 ON t.from_address = level1.node
  ),
  level2_in AS (
    SELECT t.from_address AS node
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN level1 ON t.to_address = level1.node
  ),
  level2 AS (
    SELECT node FROM level2_out
    UNION DISTINCT
    SELECT node FROM level2_in
  ),
  nodes AS (
    SELECT node FROM seed
    UNION DISTINCT
    SELECT node FROM level1
    UNION DISTINCT
    SELECT node FROM level2
  ),
  edges AS (
    SELECT
      t.from_address AS src,
      t.to_address   AS dst,
      t.value        AS amount,
      UNIX_SECONDS(t.block_timestamp) AS timestamp
    FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
    JOIN nodes ON t.from_address = nodes.node
    JOIN nodes AS n2 ON t.to_address = n2.node
  )
SELECT * FROM edges;
"""


# Записываем в файл
with open('full_gnn_query.sql', 'w') as f:
    f.write(sql)

print("Полный SQL-запрос записан в full_gnn_query.sql")

Полный SQL-запрос записан в full_gnn_query.sql


In [ ]:
# -- вставьте сюда кусок из шага 2
# WITH seed AS (
#   SELECT addr AS node
#   FROM UNNEST([
#     '0xAAA…',
#     '0xBBB…',
#     -- …
#   ]) AS addr
# ),

# -- 0–2 уровень BFS
# RECURSIVE bfs AS (
#   SELECT node, 0 AS depth FROM seed

#   UNION ALL

#   SELECT t.to_address AS node, bfs.depth + 1
#     FROM `bigquery-public-data.crypto_ethereum.transactions` t
#     JOIN bfs ON t.from_address = bfs.node
#    WHERE bfs.depth < 2

#   UNION ALL

#   SELECT t.from_address AS node, bfs.depth + 1
#     FROM `bigquery-public-data.crypto_ethereum.transactions` t
#     JOIN bfs ON t.to_address = bfs.node
#    WHERE bfs.depth < 2
# ),

# nodes AS (
#   SELECT DISTINCT node FROM bfs
# ),

# -- все рёбра внутри подграфа
# edges AS (
#   SELECT
#     t.from_address AS src,
#     t.to_address   AS dst,
#     t.value        AS amount,
#     UNIX_SECONDS(t.block_timestamp) AS timestamp
#   FROM `bigquery-public-data.crypto_ethereum.transactions` t
#   JOIN nodes n1 ON t.from_address = n1.node
#   JOIN nodes n2 ON t.to_address   = n2.node
# )

# SELECT * FROM edges;

# Новая версия

In [31]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Генерирует SQL для «ограниченного 2-hop BFS» + dry-run оценку.
"""
import pandas as pd
import subprocess
import shlex
import json
import textwrap

# ======= ПОДСТАВЬТЕ СВОЁ ======================================
PROJECT = "celtic-tendril-459507-q8"         # GCP-проект
DATASET = "phishing"          # datataset в BigQuery
LSTM_CSV = "./data/lstm.csv"          # исходник
DATE_FROM = "2023-01-01"        # начало окна
DATE_TO = "2025-04-30"        # конец окна
MAX_HOP2 = 300_000             # лимит узлов hop2
# ===============================================================

# 0) выгружаем LSTM-адреса в CSV для загрузки в BQ
df = pd.read_csv(LSTM_CSV, usecols=["Address", "FLAG"])
df["Address"] = df["Address"].str.strip().str.lower()
df.to_csv("lstm_addresses.csv", index=False)

print(f"→ Загрузите lstm_addresses.csv в `{PROJECT}.{DATASET}.lstm_addresses`"
      " (Address STRING, FLAG INT64)")

→ Загрузите lstm_addresses.csv в `celtic-tendril-459507-q8.phishing.lstm_addresses` (Address STRING, FLAG INT64)


In [ ]:

# --------------- динамически собираем SQL ---------------------
sql = f"""
-- Параметры
DECLARE date_from DATE DEFAULT '{DATE_FROM}';
DECLARE date_to   DATE DEFAULT '{DATE_TO}';
DECLARE max_hop2  INT64 DEFAULT {MAX_HOP2};

-- 0. Базовая таблица LSTM
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.lstm` AS
SELECT LOWER(Address) AS addr, CAST(FLAG AS BOOL) AS label
FROM `{PROJECT}.{DATASET}.lstm_addresses`;

-- 1. Hop-1 адреса
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.hop1` AS
SELECT DISTINCT LOWER(IF(t.from_address IN (SELECT addr FROM `{PROJECT}.{DATASET}.lstm`),
                         t.to_address, t.from_address)) AS addr
FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
WHERE DATE(block_timestamp) BETWEEN date_from AND date_to
  AND value > 0
  AND (LOWER(t.from_address) IN (SELECT addr FROM `{PROJECT}.{DATASET}.lstm`)
       OR LOWER(t.to_address)  IN (SELECT addr FROM `{PROJECT}.{DATASET}.lstm`));

-- 2. Hop-2 (с лимитом по активности)
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.hop2` AS
SELECT addr FROM (
  SELECT LOWER(IF(t.from_address IN (SELECT addr FROM `{PROJECT}.{DATASET}.hop1`),
                  t.to_address, t.from_address)) AS addr,
         COUNT(*) AS tx_cnt
  FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
  WHERE DATE(block_timestamp) BETWEEN date_from AND date_to
    AND value > 0
    AND (LOWER(t.from_address) IN (SELECT addr FROM `{PROJECT}.{DATASET}.hop1`)
         OR LOWER(t.to_address)  IN (SELECT addr FROM `{PROJECT}.{DATASET}.hop1`))
  GROUP BY addr
  QUALIFY ROW_NUMBER() OVER(ORDER BY tx_cnt DESC) <= max_hop2
);

-- 3. Итоговые узлы (метка = NULL, если адрес не в LSTM)
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.gnn_accounts` AS
SELECT addr AS id,
       ANY_VALUE(label) AS label   -- BOOL, может быть NULL
FROM (
  SELECT * FROM `{PROJECT}.{DATASET}.lstm`
  UNION DISTINCT
  SELECT addr, NULL AS label FROM `{PROJECT}.{DATASET}.hop1`
  UNION DISTINCT
  SELECT addr, NULL AS label FROM `{PROJECT}.{DATASET}.hop2`
)
GROUP BY id;

-- 4. Рёбра
CREATE OR REPLACE TABLE `{PROJECT}.{DATASET}.gnn_transactions`
PARTITION BY DATE(block_timestamp)
AS
SELECT
  LOWER(t.from_address) AS src,
  LOWER(t.to_address)   AS dst,
  SAFE_DIVIDE(CAST(t.value AS FLOAT64), 1e18) AS amount,
  t.block_timestamp
FROM `bigquery-public-data.crypto_ethereum.transactions` AS t
JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS n1
  ON LOWER(t.from_address) = n1.id
JOIN `{PROJECT}.{DATASET}.gnn_accounts` AS n2
  ON LOWER(t.to_address)   = n2.id
WHERE DATE(block_timestamp) BETWEEN date_from AND date_to
  AND value > 0;
"""

# ---- сохраняем и сразу делаем dry-run ------------------------
open("create_mini_bfs.sql", "w", encoding="UTF-8").write(sql)

2755

In [37]:
from google.cloud import bigquery

# Укажи ID проекта GCP
PROJECT = "celtic-tendril-459507-q8"

# Инициализация клиента
client = bigquery.Client(project=PROJECT)

# Чтение SQL-запроса из файла
# можно также использовать pathlib
sql_path = Path("D:/Fraud/dataset/create_mini_bfs.sql")
sql = sql_path.read_text()

# Конфигурация на dry-run
job_config = bigquery.QueryJobConfig(
    dry_run=True,
    use_query_cache=False
)

# Выполнение dry-run
query_job = client.query(sql, job_config=job_config)

# Вывод информации
print("→ Делаем dry-run…")
print("   totalBytesProcessed = {:.3f} GB".format(
    query_job.total_bytes_processed / 1e9))
if query_job.total_bytes_processed <= 5e9:
    print("   ✅ Можно запускать без риска — не превышает 5 GB.")
else:
    print("   ⚠️ Превышает 5 GB — может потребоваться включённый billing.")

Forbidden: 403 POST https://bigquery.googleapis.com/bigquery/v2/projects/celtic-tendril-459507-q8/jobs?prettyPrint=false: <!DOCTYPE html>
<html lang=en>
  <meta charset=utf-8>
  <meta name=viewport content="initial-scale=1, minimum-scale=1, width=device-width">
  <title>Error 403 (Forbidden)!!1</title>
  <style>
    *{margin:0;padding:0}html,code{font:15px/22px arial,sans-serif}html{background:#fff;color:#222;padding:15px}body{margin:7% auto 0;max-width:390px;min-height:180px;padding:30px 0 15px}* > body{background:url(//www.google.com/images/errors/robot.png) 100% 5px no-repeat;padding-right:205px}p{margin:11px 0 22px;overflow:hidden}ins{color:#777;text-decoration:none}a img{border:0}@media screen and (max-width:772px){body{background:none;margin-top:0;max-width:none;padding-right:0}}#logo{background:url(//www.google.com/images/branding/googlelogo/1x/googlelogo_color_150x54dp.png) no-repeat;margin-left:-5px}@media only screen and (min-resolution:192dpi){#logo{background:url(//www.goog

# Etherscan

In [ ]:
import time
import requests
import pandas as pd

API_KEY = ""
BASE_URL = "https://api.etherscan.io/api"

def fetch_txs(address, page=1, offset=10000):
    """Загружает транзакции (обычные) для address."""
    params = {
        "module": "account",
        "action": "txlist",
        "address": address,
        "startblock": 0,
        "endblock": 99999999,
        "page": page,
        "offset": offset,
        "sort": "desc",
        "apikey": API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    if data["status"] != "1":
        return []
    return data["result"]

def fetch_internal_txs(address, page=1, offset=10000):
    """Загружает внутренние (internal) транзакции."""
    params = {
        "module": "account",
        "action": "txlistinternal",
        "address": address,
        "startblock": 0,
        "endblock": 99999999,
        "page": page,
        "offset": offset,
        "sort": "desc",
        "apikey": API_KEY
    }
    r = requests.get(BASE_URL, params=params)
    data = r.json()
    if data["status"] != "1":
        return []
    return data["result"]

def get_all_txs(address):
    """Собирает обычные + internal, пагинируя до пустой страницы."""
    all_txs = []
    for fetcher in (fetch_txs, fetch_internal_txs):
        page = 1
        while True:
            txs = fetcher(address, page=page)
            if not txs:
                break
            all_txs.extend(txs)
            page += 1
            time.sleep(0.2)   # чтобы не превысить 5 rps
    return all_txs

# 1) Читаем LSTM-адреса
lstm = pd.read_csv("./data/transaction_dataset.csv", usecols=["Address","FLAG"])
seed_addrs = set(lstm["Address"].tolist())

# 2) BFS-обход глубины 2
visited = set(seed_addrs)
frontier = set(seed_addrs)
edges = []   # тут будем собирать ребра

for depth in range(2):
    next_frontier = set()
    for addr in frontier:
        txs = get_all_txs(addr)
        for tx in txs:
            src = tx.get("from")
            dst = tx.get("to")
            amt = tx.get("value")
            ts  = int(tx.get("timeStamp", 0))
            edges.append((src, dst, amt, ts))
            # запоминаем нового соседа
            for nbr in (src, dst):
                if nbr not in visited:
                    visited.add(nbr)
                    next_frontier.add(nbr)
    frontier = next_frontier

# 3) Сохраняем transaction.csv
tx_df = pd.DataFrame(edges, columns=["src","dst","amount","timestamp"])
tx_df.to_csv("./data/transaction.csv", index=False)

# 4) Сбор account.csv
all_nodes = pd.Series(list(tx_df["src"]) + list(tx_df["dst"]), name="id")
all_nodes = all_nodes.drop_duplicates().to_frame()
# маппим метки: LSTM→FLAG, новые узлы = -1
label_map = dict(zip(lstm["Address"], lstm["FLAG"]))
all_nodes["label"] = all_nodes["id"].map(label_map).fillna(-1).astype(int)
all_nodes.to_csv("./data/account.csv", index=False)

print("Собрано:", tx_df.shape[0], "транзакций;", all_nodes.shape[0], "узлов.")